In [4]:
import os, re, sys, math, json, time, random, glob
import numpy as np
from typing import Dict, Tuple, List, Optional
from scipy.stats import mannwhitneyu, pearsonr

# optional, for a high-quality SSIM if available
try:
    from skimage.metrics import structural_similarity as skimage_ssim
except Exception:
    skimage_ssim = None  # fall back to lightweight SSIM if skimage isn't available

project_dir = f'/home/{os.environ["USER"]}/FireScope'
os.chdir(project_dir)
if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

from config import DATA_DIR
from tqdm import tqdm

# -------------------------------
# Helpers: loading & listing
# -------------------------------

def _load_npy(path: str) -> np.ndarray:
    arr = np.load(path)
    # ensure 2D: squeeze channel if saved as (1,H,W) or (C,H,W)
    if arr.ndim == 3:
        arr = arr[0]
    elif arr.ndim > 3:
        raise ValueError(f"Unsupported array shape {arr.shape} in {path}")
    # clip/normalize defensively to [0,1]
    if not np.isfinite(arr).all():
        arr = np.nan_to_num(arr, nan=0.0, posinf=1.0, neginf=0.0)
    # If values look like [-1,1], map to [0,1]
    if arr.min() < 0.0 and arr.max() <= 1.0:
        arr = np.clip(arr * 0.5 + 0.5, 0.0, 1.0)
    else:
        arr = np.clip(arr, 0.0, 1.0)
    return arr.astype(np.float32)

def _list_npy(dir_path: str) -> List[str]:
    return sorted([f for f in os.listdir(dir_path) if f.lower().endswith(".npy")])

# -------------------------------
# Regression / similarity helpers
# -------------------------------

def _huber_loss(a: np.ndarray, b: np.ndarray, beta: float = 1.0) -> float:
    """Smooth L1 (Huber) with reduction='mean' like PyTorch SmoothL1Loss(beta)."""
    d = np.abs(a - b)
    quad = np.minimum(d, beta)
    # 0.5 * (quad^2) / beta  +  (d - quad)
    loss = 0.5 * (quad ** 2) / beta + (d - quad)
    return float(np.mean(loss))

def _mse(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.mean((a - b) ** 2))

def _mae(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.mean(np.abs(a - b)))

def _finite_diff_grads(x: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """
    Simple forward differences. x expected 2D (H, W).
    Returns (gx, gy) with the same shape as x (pad last row/col with zeros).
    """
    gx = np.zeros_like(x)
    gy = np.zeros_like(x)
    gx[:, :-1] = x[:, 1:] - x[:, :-1]
    gy[:-1, :] = x[1:, :] - x[:-1, :]
    return gx, gy

def _grad_loss(a: np.ndarray, b: np.ndarray) -> float:
    """
    L1 difference between image gradients, averaged.
    Matches the intent of many grad losses used during training.
    """
    ax, ay = _finite_diff_grads(a)
    bx, by = _finite_diff_grads(b)
    return float(np.mean(np.abs(ax - bx)) + np.mean(np.abs(ay - by)))

def _ssim(a: np.ndarray, b: np.ndarray) -> float:
    """
    SSIM in [0,1]. If skimage is available, use it; else use a lightweight fallback.
    Inputs are expected in [0,1]. Returns the SSIM (not 1-SSIM).
    """
    a = a.astype(np.float32)
    b = b.astype(np.float32)

    if skimage_ssim is not None:
        try:
            # modern skimage uses channel_axis; our arrays are 2D
            return float(skimage_ssim(a, b, data_range=1.0))
        except Exception:
            pass

    # Fallback: simplified SSIM (no Gaussian weighting)
    C1 = (0.01 ** 2)
    C2 = (0.03 ** 2)

    mu_a = np.mean(a)
    mu_b = np.mean(b)
    sigma_a2 = np.var(a)
    sigma_b2 = np.var(b)
    sigma_ab = np.mean((a - mu_a) * (b - mu_b))

    num = (2 * mu_a * mu_b + C1) * (2 * sigma_ab + C2)
    den = (mu_a ** 2 + mu_b ** 2 + C1) * (sigma_a2 + sigma_b2 + C2)
    if den == 0:
        return 1.0 if num == 0 else 0.0
    return float(num / den)

# -------------------------------
# Classification helpers (fast; no AUC)
# -------------------------------

def _flatten_valid(pred: np.ndarray, gt: np.ndarray, mask: Optional[np.ndarray] = None) -> Tuple[np.ndarray, np.ndarray]:
    """Return 1D arrays of pred probs and gt labels."""
    p = pred.ravel()
    g = gt.ravel()
    if mask is not None:
        m = mask.astype(bool).ravel()
    else:
        # valid where gt is finite (and where pred is finite)
        m = np.isfinite(g) & np.isfinite(p)
    # ensure binary labels
    # g = (g > 0.5).astype(np.uint8) if g.dtype.kind != 'b' else g.astype(np.uint8)
    return p[m], g[m]

def _f1_from_counts(tp, fp, fn):
    denom = (2*tp + fp + fn)
    return 0.0 if denom == 0 else (2*tp) / denom

def _metrics_at_threshold(scores: np.ndarray, labels: np.ndarray, thr: float):
    """Compute confusion + metrics at score >= thr => positive."""
    y_pred = (scores >= thr)
    y_true = labels.astype(bool)
    tp = int(np.sum(y_pred & y_true))
    fp = int(np.sum(y_pred & ~y_true))
    fn = int(np.sum(~y_pred & y_true))
    tn = int(np.sum(~y_pred & ~y_true))
    acc = (tp + tn) / max(1, len(labels))
    f1 = _f1_from_counts(tp, fp, fn)
    return tp, fp, fn, tn, acc, f1

def _binary_confusion(y_true: np.ndarray, y_prob: np.ndarray, thr: float) -> Dict[str, float]:
    y_pred = (y_prob >= thr).astype(np.uint8)
    tp = int(np.sum((y_pred == 1) & (y_true == 1)))
    tn = int(np.sum((y_pred == 0) & (y_true == 0)))
    fp = int(np.sum((y_pred == 1) & (y_true == 0)))
    fn = int(np.sum((y_pred == 0) & (y_true == 1)))
    prec = tp / max(tp + fp, 1)
    rec = tp / max(tp + fn, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-12)
    iou = tp / max(tp + fp + fn, 1)
    acc = (tp + tn) / max(tp + tn + fp + fn, 1)
    tnr = tn / max(tn + fp, 1)  # specificity
    return dict(tp=tp, tn=tn, fp=fp, fn=fn, precision=prec, recall=rec, f1=f1, iou=iou, accuracy=acc, specificity=tnr)

def _brier_score(probs: np.ndarray, labels: np.ndarray) -> float:
    """Mean squared error between probabilities and binary labels."""
    l = labels.astype(np.float32)
    p = probs.astype(np.float32)
    return float(np.mean((p - l) ** 2))

def _mcc_from_counts(tp: int, fp: int, fn: int, tn: int) -> float:
    """Matthews Correlation Coefficient from confusion counts."""
    num = (tp * tn) - (fp * fn)
    den = math.sqrt(max((tp + fp), 1) * max((tp + fn), 1) * max((tn + fp), 1) * max((tn + fn), 1))
    return 0.0 if den == 0 else float(num / den)

def _cohen_kappa_from_counts(tp: int, fp: int, fn: int, tn: int) -> float:
    """Cohen's kappa from confusion counts."""
    total = tp + fp + fn + tn
    if total == 0:
        return float('nan')
    po = (tp + tn) / total
    # expected agreement by chance from marginals
    p_yes_pred = (tp + fp) / total
    p_yes_true = (tp + fn) / total
    p_no_pred  = (tn + fn) / total
    p_no_true  = (tn + fp) / total
    pe = p_yes_pred * p_yes_true + p_no_pred * p_no_true
    denom = (1.0 - pe)
    return 0.0 if denom == 0 else float((po - pe) / denom)

def _roc_auc_from_scores(y: np.ndarray, p: np.ndarray) -> float:
    """ROC AUC via Mann–Whitney: P(p_pos > p_neg)."""
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    pos = p[y == 1]
    neg = p[y == 0]
    if len(pos) == 0 or len(neg) == 0:
        return np.nan
    U = mannwhitneyu(pos, neg, alternative="greater", method="asymptotic").statistic
    return float(U / (len(pos) * len(neg)))

def _brier(y: np.ndarray, p: np.ndarray) -> float:
    y = np.asarray(y, dtype=float); p = np.asarray(p, dtype=float)
    return float(np.mean((p - y) ** 2))

def _logloss(y: np.ndarray, p: np.ndarray, eps: float = 1e-12) -> float:
    y = np.asarray(y, dtype=float); p = np.asarray(p, dtype=float)
    p = np.clip(p, eps, 1.0 - eps)
    return float(-np.mean(y * np.log(p) + (1.0 - y) * np.log(1.0 - p)))

def _pearson_corr(y: np.ndarray, p: np.ndarray) -> float:
    # Point-biserial correlation equals Pearson between binary y and continuous p
    if len(y) < 2 or np.all(y == y[0]):
        return np.nan
    r, _ = pearsonr(p, y)
    return float(r)

def _ece_mce(y: np.ndarray, p: np.ndarray, n_bins: int = 15) -> tuple[float, float]:
    """
    Expected Calibration Error (ECE) and Max Calibration Error (MCE).
    Bins are linear in [0,1]. Returns (ECE, MCE).
    """
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    # Guard: if predictions are not in [0,1], clip (your loader already clamps)
    p = np.clip(p, 0.0, 1.0)
    N = len(p)
    if N == 0:
        return np.nan, np.nan

    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    mce = 0.0
    for b in range(n_bins):
        lo, hi = edges[b], edges[b+1]
        mask = (p >= lo) & (p < hi) if b < n_bins - 1 else (p >= lo) & (p <= hi)
        nb = int(mask.sum())
        if nb == 0:
            continue
        conf = float(p[mask].mean())
        acc = float(y[mask].mean())  # observed frequency
        gap = abs(acc - conf)
        ece += (nb / N) * gap
        mce = max(mce, gap)
    return float(ece), float(mce)

def _append_classification_metrics(out: Dict[str, float],
                                   all_probs: np.ndarray,
                                   all_labels: np.ndarray,
                                   threshold: float = 0.5,
                                   verbose: bool = False, n_bins=15) -> Dict[str, float]:
    """
    Fast classification/probabilistic metrics (no ROC/PR curves, no AUC):
      - Brier score
      - Confusion-matrix metrics @ threshold: precision, recall, F1, IoU, accuracy, specificity
      - MCC, Cohen's kappa
    """
    # Confusion-based metrics @ threshold
    tp, fp, fn, tn, acc, f1 = _metrics_at_threshold(all_probs, all_labels, threshold)
    cm = _binary_confusion((all_labels > threshold).astype(np.uint8), all_probs, threshold)

    mcc = _mcc_from_counts(tp, fp, fn, tn)
    kappa = _cohen_kappa_from_counts(tp, fp, fn, tn)

    y = all_labels
    p = all_probs
    r = _pearson_corr(y, p)
    br = _brier(y, p)
    ll = _logloss(y, (all_labels > threshold).astype(np.uint8))
    auc = _roc_auc_from_scores((all_labels > threshold).astype(np.uint8), p)
    ece, mce = _ece_mce(y, p, n_bins=n_bins)

    out.update({
        "Correlation": float(r),
        "Brier": float(br),
        "LogLoss": float(ll),
        "ROC_AUC": float(auc),
        "ECE": float(ece),
        "Reliability(MCE)": float(mce),
        "precision@thr": float(cm["precision"]),
        "recall@thr": float(cm["recall"]),
        "f1@thr": float(cm["f1"]),
        "iou@thr": float(cm["iou"]),
        "accuracy@thr": float(cm["accuracy"]),
        "specificity@thr": float(cm["specificity"]),
        "mcc": float(mcc),
        "cohen_kappa": float(kappa),
        "threshold_used": float(threshold),
    })

    if verbose:
        print("---- Classification metrics (fast) ----")
        print(f"Correlation        : {out['Correlation']:.6f}")
        print(f"Brier Score        : {out['brier_score']:.6f}")
        print(f"Threshold          : {threshold:.3f}")
        print(f"Precision@thr      : {out['precision@thr']:.6f}")
        print(f"Recall@thr         : {out['recall@thr']:.6f}")
        print(f"F1@thr             : {out['f1@thr']:.6f}")
        print(f"IoU@thr            : {out['iou@thr']:.6f}")
        print(f"Accuracy@thr       : {out['accuracy@thr']:.6f}")
        print(f"Specificity@thr    : {out['specificity@thr']:.6f}")
        print(f"MCC                : {out['mcc']:.6f}")
        print(f"Cohen's kappa      : {out['cohen_kappa']:.6f}")

    return out

# -------------------------------
# Public API
# -------------------------------

def get_metrics(pred_path, gt_path, add_curves: bool = True, threshold: float = 0.5, mask_dir: Optional[str] = None) -> Dict[str, float]:
    """
    Extended version (fast):
      - add_curves: if True, computes fast classification metrics (no ROC/PR curves, no AUC)
      - threshold:  cutoff for binarizing predictions (for confusion-matrix metrics)
      - mask_dir:   optional directory of boolean masks (same filenames) where True=valid pixels
    """
    pred_items = _list_npy(pred_path)
    gt_items = _list_npy(gt_path)
    for item in pred_items:
        if item not in gt_items:
            raise Exception(f'{item} found in predictions but not in ground truth!')
    if len(pred_items) == 0:
        raise ValueError("No .npy files found.")

    totals = {"recon": 0.0, "ssim": 0.0, "grad": 0.0, "mse": 0.0, "mae": 0.0}
    n = 0
    all_probs_list, all_labels_list = [], []

    for name in tqdm(pred_items):
        pred = _load_npy(os.path.join(pred_path, name))  # [0,1] prob map
        gt   = _load_npy(os.path.join(gt_path, name))    # assumed binary (0/1) or continuous then binarized (>0.5)

        if pred.shape != gt.shape:
            raise ValueError(f"Shape mismatch for {name}: pred {pred.shape} vs gt {gt.shape}")

        # optional validity mask
        mask = None
        if mask_dir:
            mask_fp = os.path.join(mask_dir, name)
            if os.path.exists(mask_fp):
                mask = np.load(mask_fp).astype(bool)
                if mask.shape != pred.shape:
                    raise ValueError(f"Mask shape mismatch for {name}: {mask.shape} vs {pred.shape}")

        # --- regression-style losses on continuous maps ---
        l_recon = _huber_loss(pred, gt)
        ssim_val = _ssim(pred, gt)
        l_ssim = 1.0 - ssim_val
        l_grad = _grad_loss(pred, gt)
        l_mse = _mse(pred, gt)
        l_mae = _mae(pred, gt)

        totals["recon"] += l_recon
        totals["ssim"]  += l_ssim
        totals["grad"]  += l_grad
        totals["mse"]   += l_mse
        totals["mae"]   += l_mae
        n += 1

        # --- collect for fast classification metrics (prob/label vectors) ---
        if add_curves:
            probs_flat, labels_flat = _flatten_valid(pred, gt, mask)
            all_probs_list.append(probs_flat)
            all_labels_list.append(labels_flat)

    out = {k: (v / max(n, 1)) for k, v in totals.items()}

    # Add fast classification metrics (no AUC/curves)
    if add_curves and all_probs_list:
        all_probs = np.concatenate(all_probs_list)
        all_labels = np.concatenate(all_labels_list)
        out = _append_classification_metrics(out, all_probs, all_labels,
                                             threshold=threshold, verbose=False)

    return out

In [ ]:
# --- Multi-folder evaluation + pretty table with bold best-per-metric ---

import os
import pandas as pd
from typing import List, Optional, Dict

# Assumes get_metrics(...) is already defined in the notebook (from your code)

def evaluate_prediction_folders(
    gt_dir: str,
    pred_dirs: List[str],
    threshold: float = 0.5,
    mask_dir: Optional[str] = None,
    add_classification_metrics: bool = True,
    df_labels = None
) -> pd.DataFrame:
    """
    Runs get_metrics for each prediction folder and returns a DataFrame with rows = model (folder name),
    columns = metrics. Folder names are basename-only (no full paths).
    """
    rows: List[Dict[str, float]] = []
    names: List[str] = []

    for pdir in pred_dirs:
        name = os.path.basename(os.path.dirname(os.path.normpath(pdir))) or pdir
        names.append(name)
        out = get_metrics(
            pred_path=pdir,
            gt_path=gt_dir,
            add_curves=add_classification_metrics,  # in your code: this means "add fast classification metrics"
            threshold=threshold,
            mask_dir=mask_dir,
        )
        rows.append(out)

    if df_labels and len(df_labels) == len(names):
        names = df_labels
    df = pd.DataFrame(rows, index=names)

    return df


def style_bold_best(df: pd.DataFrame, precision=4):
    """
    Bold the best value per column:
      - For 'higher is better' metrics: bold the max.
      - For 'lower is better' metrics: bold the min.
    """
    higher_is_better = {"Correlation", "ROC_AUC", "Delta", "Precision", "Recall", "f1", "IoU",
                    "accuracy", "specificity", "mcc", "cohen_kappa"}
    lower_is_better  = {"Brier", "LogLoss", "ECE", "Reliability(MCE)", "recon", "ssim", "grad", "mse", "mae"}


    def highlight(col: pd.Series) -> list:
        if col.name in higher_is_better:
            best = col.max(skipna=True)
            return ["font-weight: bold" if v == best else "" for v in col]
        elif col.name in lower_is_better:
            best = col.min(skipna=True)
            return ["font-weight: bold" if v == best else "" for v in col]
        else:
            return [""] * len(col)

    return (
        df.style
          .apply(highlight, axis=0)
          .format(precision=precision)
    )


threshold = 0.5
mask_dir = None  # or "/path/to/masks" if you have valid masks
gt_dir = f"{DATA_DIR}/large_dataset/normalised_risk_rasters/test"
pred_dirs = {
    'unet_baseline':f"{DATA_DIR}/evaluations/small/unet_baseline/europe_fires",
    'unet_climate':f"{DATA_DIR}/evaluations/small/unet_climate/europe_fires",
    'unet_oracle':f"{DATA_DIR}/evaluations/small/oracle_unet_no_cot/europe_fires",
    'unet_cot_oracle':f"{DATA_DIR}/evaluations/small/oracle_unet/europe_fires",

    'alphaearth_baseline':f"{DATA_DIR}/evaluations/small/alphaearth_baseline/europe_fires_ada",
    'alphaearth_climate':f"{DATA_DIR}/evaluations/small/alphaearth_climate/europe_fires_ada",
    'alphaearth_oracle':f"{DATA_DIR}/evaluations/small/oracle_alphaearth_no_cot/europe_fires_ada",
    'alphaearth_cot_oracle':f"{DATA_DIR}/evaluations/small/oracle_alphaearth/europe_fires_ada",

    'segformer_baseline':f"{DATA_DIR}/evaluations/small/segformer_baseline/europe_fires",
    'segformer_climate':f"{DATA_DIR}/evaluations/small/segformer_climate/europe_fires",
    'segformer_oracle':f"{DATA_DIR}/evaluations/small/oracle_segformer_no_cot/europe_fires",
    'segformer_cot_oracle':f"{DATA_DIR}/evaluations/small/oracle_segformer/europe_fires",

    'qwen-decoder':f"{DATA_DIR}/evaluations/small/qwen_vlm_decoder_head/europe_fires",
    'unet^_baseline':f"{DATA_DIR}/evaluations/small/big/unet_baseline/europe_fires",
    'unet^_climate':f"{DATA_DIR}/evaluations/small/big/unet_climate/europe_fires",
}
df = evaluate_prediction_folders(gt_dir, list(pred_dirs.values()), threshold=threshold, mask_dir=mask_dir, df_labels=list(pred_dirs.keys()))
styler = style_bold_best(df)
display(styler)
